# Preprocessing

Import potrzebnych bibliotek

In [1]:
import pandas as pd
import numpy as np
import time

Wczytanie danych

In [2]:
df = pd.read_csv("auction_results_color_svd.csv")

df.head()

,ARTIST,TECHNIQUE,SIGNATURE,CONDITION,TOTAL DIMENSIONS,YEAR,Colorfulness Score,SVD Entropy,PRICE
0,218,-1.295300,1,2,-0.157723,-1.039766,51.632554,5.453204,150
1,101,-0.122087,2,2,-0.442668,-0.580107,161.631656,6.154763,270
2,274,-0.122087,2,2,0.263423,-0.626073,117.464780,6.908661,360
3,354,3.397553,0,2,-0.827075,-0.488176,164.609302,6.986244,343
4,354,-0.122087,2,2,-0.145178,0.431142,91.023011,5.859255,150


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22110 entries, 0 to 22109
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ARTIST              22110 non-null  int64  
 1   TECHNIQUE           22110 non-null  float64
 2   SIGNATURE           22110 non-null  int64  
 3   CONDITION           22110 non-null  int64  
 4   TOTAL DIMENSIONS    22110 non-null  float64
 5   YEAR                22110 non-null  float64
 6   Colorfulness Score  22110 non-null  float64
 7   SVD Entropy         22110 non-null  float64
 8   PRICE               22110 non-null  int64  
dtypes: float64(5), int64(4)
memory usage: 1.5 MB


Dopasowanie typów zmiennych

In [4]:
zmienne_kategoryczne =  ['ARTIST', 'TECHNIQUE', 'SIGNATURE', 'CONDITION']

bloki_kategoryczne = []
for zmienna in zmienne_kategoryczne:
    kody = df[zmienna].astype('category').cat.codes.to_numpy(dtype=np.int32)
    liczba_klas = np.max(kody) + 1
    one_hot = np.eye(liczba_klas)[kody]
    bloki_kategoryczne.append(one_hot)


X_kat_cale = np.hstack(bloki_kategoryczne)


Dane treningowe i testowe

In [5]:
np.random.seed(42)
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

indeksy_train = df_train.index
indeksy_test = df_test.index

X_kat_train = X_kat_cale[indeksy_train]
X_kat_test = X_kat_cale[indeksy_test]

Normalizacja zmiennych

In [6]:
zmienne_liczbowe = ["TOTAL DIMENSIONS", "YEAR", "Colorfulness Score", "SVD Entropy"]
srednia = df_train[zmienne_liczbowe].mean()
odchylenie = df_train[zmienne_liczbowe].std()

df_train[zmienne_liczbowe] = (df_train[zmienne_liczbowe] - srednia) / odchylenie
df_test[zmienne_liczbowe] = (df_test[zmienne_liczbowe] - srednia) / odchylenie

Przejście na tablice NumPy

In [7]:
X_num_train = df_train[zmienne_liczbowe].to_numpy(dtype=np.float32)
X_num_test = df_test[zmienne_liczbowe].to_numpy(dtype=np.float32)

X_train = np.hstack([X_kat_train, X_num_train], dtype=np.float32)
X_test = np.hstack([X_kat_test, X_num_test], dtype=np.float32)


# Tworzenie sieci

Tworzenie Dense Layer

In [8]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        # Klasyczna inicjalizacja wag
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases
        
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)

Funkcje aktywacji

In [ ]:
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

class Activation_Linear:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = inputs
    
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
    
    
class Layer_Dropout:
    # rate to ułamek neuronów, które CHCEMY WYŁĄCZYĆ (np. 0.2 oznacza wyłączenie 20%)
    def __init__(self, rate):
        self.rate = 1 - rate # Zapisujemy wskaźnik zachowania (

    def forward(self, inputs, training=True):
        self.inputs = inputs
        
        # Jeśli testujemy model, wyłączamy Dropout (przepuszczamy dane 1:1)
        if not training:
            self.output = inputs.copy()
            return
            
        # Generujemy maskę (0 lub 1) i od razu skalujemy (Inverted Dropout)
        self.binary_mask = np.random.binomial(1, self.rate, size=inputs.shape) / self.rate
        self.output = inputs * self.binary_mask

    def backward(self, dvalues):
        # Wsteczna propagacja po prostu przepuszcza gradienty tylko tam, gdzie maska ma 1
        self.dinputs = dvalues * self.binary_mask

Funkcja straty

In [34]:
class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss
    
class Loss_MSE(Loss):
    def forward(self, y_pred, y_true):
        sample_losses = np.mean((y_pred - y_true) ** 2, axis=-1)
        return sample_losses
    
    def backward(self, dvalues, y_true):
        liczba_probek = len(dvalues)
        self.dinputs = -2 * (y_true - dvalues) / liczba_probek



Optymalizator SGD

In [11]:
class Optimizer_SGD:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update_params(self, layer):
        layer.weights -= self.learning_rate * layer.dweights
        layer.biases -= self.learning_rate * layer.dbiases

# Model Bazowy

In [ ]:

print("="*70)
print("BEZPOŚREDNIE PORÓWNANIE: Z-SCORE vs LOGARYTM (Model Bazowy, Średnia z 3 prób)")
print("="*70)

# ---------------------------------------------------------
# 1. PRZYGOTOWANIE OBU ZESTAWÓW DANYCH
# ---------------------------------------------------------
# Wersja A: Z-Score
srednia_cena = df_train['PRICE'].mean()
odchylenie_cena = df_train['PRICE'].std()

y_train_z = ((df_train['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
y_test_z = ((df_test['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)

def prawdziwa_cena_z(znormalizowana_cena):
    return znormalizowana_cena * odchylenie_cena + srednia_cena

# Wersja B: Transformacja Logarytmiczna
y_train_log = np.log(df_train['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)
y_test_log = np.log(df_test['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)

def prawdziwa_cena_log(wyliczona_wartosc):
    return np.exp(wyliczona_wartosc)


metody_testowe = [
    ("Normalizacja Z-Score", y_train_z, y_test_z, prawdziwa_cena_z),
    ("Transformacja Logarytmiczna", y_train_log, y_test_log, prawdziwa_cena_log)
]

# Domyślne parametry modelu bazowego
n1, n2 = 64, 32
domyslny_lr = 0.1
domyslny_batch = 256
epoki = 100
powtorzenia = 3  

# ---------------------------------------------------------
# 2. PĘTLA TESTUJĄCA OBIE METODY
# ---------------------------------------------------------
for nazwa_metody, y_train_aktualne, y_test_aktualne, funkcja_odwracajaca in metody_testowe:
    print(f"\n---> ROZPOCZYNAM TRENOWANIE DLA: {nazwa_metody} ...")
    
    # Listy do agregacji wyników z 3 prób
    tr_mae_list, tr_rmse_list, tr_r2_list, tr_mape_list, tr_smape_list = [], [], [], [], []
    te_mae_list, te_rmse_list, te_r2_list, te_mape_list, te_smape_list = [], [], [], [], []

    for p in range(powtorzenia):
        print(f"     Próba {p+1}/{powtorzenia}...", end=" ")
        start_time = time.time()
        
        # Świeża inicjalizacja sieci dla każdej próby!
        dense1_b = Layer_Dense(X_train.shape[1], n1)
        activation1_b = Activation_ReLU()

        dense2_b = Layer_Dense(n1, n2)
        activation2_b = Activation_ReLU()

        dense3_b = Layer_Dense(n2, 1)
        activation3_b = Activation_Linear()

        loss_function_b = Loss_MSE()
        optimizer_b = Optimizer_SGD(learning_rate=domyslny_lr)

        # Trening
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), domyslny_batch):
                end_idx = start_idx + domyslny_batch
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train_aktualne[start_idx:end_idx]
                
                dense1_b.forward(X_batch)
                activation1_b.forward(dense1_b.output)
                dense2_b.forward(activation1_b.output)
                activation2_b.forward(dense2_b.output)
                dense3_b.forward(activation2_b.output)
                activation3_b.forward(dense3_b.output)
                
                loss_function_b.backward(activation3_b.output, y_batch)
                activation3_b.backward(loss_function_b.dinputs)
                dense3_b.backward(activation3_b.dinputs)
                activation2_b.backward(dense3_b.dinputs)
                dense2_b.backward(activation2_b.dinputs)
                activation1_b.backward(dense2_b.dinputs)
                dense1_b.backward(activation1_b.dinputs)
                
                optimizer_b.update_params(dense1_b)
                optimizer_b.update_params(dense2_b)
                optimizer_b.update_params(dense3_b)

        # Ewaluacja (Train)
        dense1_b.forward(X_train)
        activation1_b.forward(dense1_b.output)
        dense2_b.forward(activation1_b.output)
        activation2_b.forward(dense2_b.output)
        dense3_b.forward(activation2_b.output)
        activation3_b.forward(dense3_b.output)

        wymyslone_ceny_train = funkcja_odwracajaca(activation3_b.output)
        prawdziwe_ceny_train = funkcja_odwracajaca(y_train_aktualne)

        mae_train = np.mean(np.abs(prawdziwe_ceny_train - wymyslone_ceny_train))
        rmse_train = np.sqrt(np.mean((prawdziwe_ceny_train - wymyslone_ceny_train)**2))
        ss_res_train = np.sum((prawdziwe_ceny_train - wymyslone_ceny_train)**2)
        ss_tot_train = np.sum((prawdziwe_ceny_train - np.mean(prawdziwe_ceny_train))**2)
        r2_train = 1 - (ss_res_train / ss_tot_train)

        non_zero_train = prawdziwe_ceny_train != 0
        mape_train = np.mean(np.abs((prawdziwe_ceny_train[non_zero_train] - wymyslone_ceny_train[non_zero_train]) / prawdziwe_ceny_train[non_zero_train])) * 100
        
        licznik_tr = np.abs(prawdziwe_ceny_train - wymyslone_ceny_train)
        mianownik_tr = (np.abs(prawdziwe_ceny_train) + np.abs(wymyslone_ceny_train)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100

        tr_mae_list.append(mae_train)
        tr_rmse_list.append(rmse_train)
        tr_r2_list.append(r2_train)
        tr_mape_list.append(mape_train)
        tr_smape_list.append(smape_train)

        # Ewaluacja (Test)
        dense1_b.forward(X_test)
        activation1_b.forward(dense1_b.output)
        dense2_b.forward(activation1_b.output)
        activation2_b.forward(dense2_b.output)
        dense3_b.forward(activation2_b.output)
        activation3_b.forward(dense3_b.output)

        wymyslone_ceny_test = funkcja_odwracajaca(activation3_b.output)
        prawdziwe_ceny_test = funkcja_odwracajaca(y_test_aktualne)

        mae_test = np.mean(np.abs(prawdziwe_ceny_test - wymyslone_ceny_test))
        rmse_test = np.sqrt(np.mean((prawdziwe_ceny_test - wymyslone_ceny_test)**2))
        ss_res_test = np.sum((prawdziwe_ceny_test - wymyslone_ceny_test)**2)
        ss_tot_test = np.sum((prawdziwe_ceny_test - np.mean(prawdziwe_ceny_test))**2)
        r2_test = 1 - (ss_res_test / ss_tot_test)

        non_zero_test = prawdziwe_ceny_test != 0
        mape_test = np.mean(np.abs((prawdziwe_ceny_test[non_zero_test] - wymyslone_ceny_test[non_zero_test]) / prawdziwe_ceny_test[non_zero_test])) * 100

        licznik_te = np.abs(prawdziwe_ceny_test - wymyslone_ceny_test)
        mianownik_te = (np.abs(prawdziwe_ceny_test) + np.abs(wymyslone_ceny_test)) 
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 200

        te_mae_list.append(mae_test)
        te_rmse_list.append(rmse_test)
        te_r2_list.append(r2_test)
        te_mape_list.append(mape_test)
        te_smape_list.append(smape_test)
        
        print(f"Zakończono w {time.time() - start_time:.1f} s")

    # Wyświetlanie UŚREDNIONYCH wyników
    print("-" * 70)
    print(f"ŚREDNIE WYNIKI BAZOWE (z {powtorzenia} prób): {nazwa_metody}")
    print("-" * 70)
    print(f"ZBIÓR TRENINGOWY:")
    print(f"  MAE: {np.mean(tr_mae_list):10.2f} $ | MAPE: {np.mean(tr_mape_list):6.2f} % | sMAPE: {np.mean(tr_smape_list):6.2f} % | RMSE: {np.mean(tr_rmse_list):10.2f} $ | R^2: {np.mean(tr_r2_list):6.4f}")
    print(f"ZBIÓR TESTOWY:")
    print(f"  MAE: {np.mean(te_mae_list):10.2f} $ | MAPE: {np.mean(te_mape_list):6.2f} % | sMAPE: {np.mean(te_smape_list):6.2f} % | RMSE: {np.mean(te_rmse_list):10.2f} $ | R^2: {np.mean(te_r2_list):6.4f}")

# ---------------------------------------------------------
# 3. ZATWIERDZENIE ZWYCIĘSKIEJ METODY NA RESZTĘ KODU
# ---------------------------------------------------------
print("\n" + "="*70)
print("Zakończono porównanie. Metoda Logarytmiczna okazała się lepsza.")
print("Ustawiam dane logarytmiczne jako domyślne dla dalszej optymalizacji sieci.")
print("="*70)

y_train = y_train_log
y_test = y_test_log
prawdziwa_cena = prawdziwa_cena_log

BEZPOŚREDNIE PORÓWNANIE: Z-SCORE vs LOGARYTM (Model Bazowy, Średnia z 3 prób)

---> ROZPOCZYNAM TRENOWANIE DLA: Normalizacja Z-Score ...
     Próba 1/3... Zakończono w 4.7 s
     Próba 2/3... Zakończono w 5.1 s
     Próba 3/3... Zakończono w 4.5 s
----------------------------------------------------------------------
ŚREDNIE WYNIKI BAZOWE (z 3 prób): Normalizacja Z-Score
----------------------------------------------------------------------
ZBIÓR TRENINGOWY:
  MAE:     104.60 $ | MAPE: 219.04 % | sMAPE:  69.17 % | RMSE:     275.50 $ | R^2: 0.6721
ZBIÓR TESTOWY:
  MAE:     113.14 $ | MAPE: 245.74 % | sMAPE:  70.30 % | RMSE:     290.86 $ | R^2: 0.6175

---> ROZPOCZYNAM TRENOWANIE DLA: Transformacja Logarytmiczna ...
     Próba 1/3... Zakończono w 4.7 s
     Próba 2/3... Zakończono w 4.9 s
     Próba 3/3... Zakończono w 4.6 s
----------------------------------------------------------------------
ŚREDNIE WYNIKI BAZOWE (z 3 prób): Transformacja Logarytmiczna
--------------------------------

# Badanie parametrów

Badanie liczby neuronów

In [ ]:

# 5 różnych wariantów liczb neuronów (warstwa1, warstwa2)
architektury_do_testu = [
    (32, 16),
    (64, 32),
    (128, 64),
    (256, 128),
    (512, 256)
]
powtorzenia = 3       
epoki = 100           
batch_size = 256
learning_rate = 0.1  

wyniki_raport = []
liczba_cech = X_train.shape[1]

print("="*70)
print("ROZPOCZYNAMY BADANIE: Rozmiar warstw ukrytych (Metryka: sMAPE)")
print("="*70)

for arch in architektury_do_testu:
    n1, n2 = arch
    print(f"\n---> Testuję architekturę: [{n1}, {n2}] Pętla powtórzeń: ", end="")
    
    smape_train_historia = []
    smape_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=learning_rate)
        
        # TRENING
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), batch_size):
                end_idx = start_idx + batch_size
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # EWALUACJA (Train)
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        
        licznik_tr = np.abs(prawdziwe_train_dolary - wymyslone_train_dolary)
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + np.abs(wymyslone_train_dolary)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # EWALUACJA (Test)
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        
        licznik_te = np.abs(prawdziwe_test_dolary - wymyslone_test_dolary)
        mianownik_te = (np.abs(prawdziwe_test_dolary) + np.abs(wymyslone_test_dolary)) / 2.0
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100
        smape_test_historia.append(smape_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    sredni_czas_proby = czas_trwania / powtorzenia
    
    
    wyniki_raport.append({
        "Architektura": f"[{n1}, {n2}]",
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

# WYŚWIETLANIE TABELKI
print("\n" + "="*70)
print("PODSUMOWANIE WYNIKÓW: WPŁYW ARCHITEKTURY")
print("="*70)
df_raport = pd.DataFrame(wyniki_raport)
print(df_raport.to_string(index=False))

ROZPOCZYNAMY BADANIE: Rozmiar warstw ukrytych (Metryka: sMAPE)

---> Testuję architekturę: [32, 16] Pętla powtórzeń: 1... 2... 3... Zakończono w 7.4 s.

---> Testuję architekturę: [64, 32] Pętla powtórzeń: 1... 2... 3... Zakończono w 8.9 s.

---> Testuję architekturę: [128, 64] Pętla powtórzeń: 1... 2... 3... Zakończono w 14.3 s.

---> Testuję architekturę: [256, 128] Pętla powtórzeń: 1... 2... 3... Zakończono w 27.3 s.

---> Testuję architekturę: [512, 256] Pętla powtórzeń: 1... 2... 3... Zakończono w 59.4 s.

PODSUMOWANIE WYNIKÓW: WPŁYW ARCHITEKTURY
Architektura  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
    [32, 16]          2.48                56.90               60.67                     58.71
    [64, 32]          2.96                54.43               59.02                     57.62
   [128, 64]          4.77                54.29               59.41                     57.95
  [256, 128]          9.08                52.25               58.1

Badanie Learning rate

In [17]:
n1, n2 = 256, 128

# 4 warianty współczynnika uczenia do przetestowania:
learning_rates_do_testu = [0.1, 0.05, 0.01, 0.001]

powtorzenia = 3       
epoki = 100           
batch_size = 256      

wyniki_raport_lr = []
liczba_cech = X_train.shape[1]

print("="*70)
print(f"ROZPOCZYNAMY BADANIE: Współczynnik uczenia (Sieć: [{n1}, {n2}], Metryka: sMAPE)")
print("="*70)

for lr in learning_rates_do_testu:
    print(f"\n---> Testuję Learning Rate: {lr} Pętla powtórzeń: ", end="")
    
    smape_train_historia = []
    smape_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=lr) 
        
        # TRENING
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), batch_size):
                end_idx = start_idx + batch_size
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # EWALUACJA (Train)
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        
        licznik_tr = np.abs(prawdziwe_train_dolary - wymyslone_train_dolary)
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + np.abs(wymyslone_train_dolary)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # EWALUACJA (Test)
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        
        licznik_te = np.abs(prawdziwe_test_dolary - wymyslone_test_dolary)
        mianownik_te = (np.abs(prawdziwe_test_dolary) + np.abs(wymyslone_test_dolary)) / 2.0
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100
        smape_test_historia.append(smape_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    sredni_czas_proby = czas_trwania / powtorzenia

    # AGREGACJA ZGODNIE Z NOWYM FORMATEM
    wyniki_raport_lr.append({
        "Learning Rate": lr,
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

# WYŚWIETLANIE TABELKI
print("\n" + "="*70)
print("PODSUMOWANIE WYNIKÓW: WPŁYW WSPÓŁCZYNNIKA UCZENIA (LR)")
print("="*70)
df_raport_lr = pd.DataFrame(wyniki_raport_lr)
print(df_raport_lr.to_string(index=False))

ROZPOCZYNAMY BADANIE: Współczynnik uczenia (Sieć: [256, 128], Metryka: sMAPE)

---> Testuję Learning Rate: 0.1 Pętla powtórzeń: 1... 2... 3... Zakończono w 28.4 s.

---> Testuję Learning Rate: 0.05 Pętla powtórzeń: 1... 2... 3... Zakończono w 27.3 s.

---> Testuję Learning Rate: 0.01 Pętla powtórzeń: 1... 2... 3... Zakończono w 28.3 s.

---> Testuję Learning Rate: 0.001 Pętla powtórzeń: 1... 2... 3... Zakończono w 26.5 s.

PODSUMOWANIE WYNIKÓW: WPŁYW WSPÓŁCZYNNIKA UCZENIA (LR)
 Learning Rate  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
         0.100          9.46                53.96               59.60                     58.45
         0.050          9.09                55.39               59.21                     57.62
         0.010          9.44                56.20               58.38                     58.33
         0.001          8.83                70.84               70.62                     70.54


Badanie Batch Size

In [ ]:

n1, n2 = 256, 128
najlepszy_lr = 0.01  

# 4 warianty wielkości paczki do przetestowania:
batch_sizes_do_testu = [32, 64, 128, 256]

powtorzenia = 3       
epoki = 100           

wyniki_raport_batch = []
liczba_cech = X_train.shape[1]

print("="*70)
print(f"ROZPOCZYNAMY BADANIE: Wielkość paczki (Sieć: [{n1}, {n2}], LR: {najlepszy_lr}, Metryka: sMAPE)")
print("="*70)

for b_size in batch_sizes_do_testu:
    print(f"\n---> Testuję Batch Size: {b_size:3} Pętla powtórzeń: ", end="")
    
    smape_train_historia = []
    smape_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        # 1. INICJUJEMY SIEĆ
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 
        
        # 2. TRENING 
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), b_size):
                end_idx = start_idx + b_size
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                # Forward
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                # Błąd i Backward
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                # Aktualizacja
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # 3. EWALUACJA (Train)
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        
        licznik_tr = np.abs(prawdziwe_train_dolary - wymyslone_train_dolary)
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + wymyslone_train_dolary) / 2.0 # korekta dla w. bezwzględnej dodana niżej
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + np.abs(wymyslone_train_dolary)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # 4. EWALUACJA (Test)
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        
        licznik_te = np.abs(prawdziwe_test_dolary - wymyslone_test_dolary)
        mianownik_te = (np.abs(prawdziwe_test_dolary) + np.abs(wymyslone_test_dolary)) / 2.0
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100
        smape_test_historia.append(smape_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    sredni_czas_proby = czas_trwania / powtorzenia

    # 5. AGREGACJA
    wyniki_raport_batch.append({
        "Batch Size": b_size,
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

# --- WYSWIETLANIE TABELKI ---
print("\n" + "="*70)
print("PODSUMOWANIE WYNIKÓW: WPŁYW ROZMIARU PACZKI (BATCH SIZE)")
print("="*70)
df_raport_batch = pd.DataFrame(wyniki_raport_batch)
print(df_raport_batch.to_string(index=False))

ROZPOCZYNAMY BADANIE: Wielkość paczki (Sieć: [256, 128], LR: 0.01, Metryka: sMAPE)

---> Testuję Batch Size:  32 Pętla powtórzeń: 1... 2... 3... Zakończono w 51.3 s.

---> Testuję Batch Size:  64 Pętla powtórzeń: 1... 2... 3... Zakończono w 36.8 s.

---> Testuję Batch Size: 128 Pętla powtórzeń: 1... 2... 3... Zakończono w 30.9 s.

---> Testuję Batch Size: 256 Pętla powtórzeń: 1... 2... 3... Zakończono w 26.9 s.

PODSUMOWANIE WYNIKÓW: WPŁYW ROZMIARU PACZKI (BATCH SIZE)
 Batch Size  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
         32         17.11                51.40               58.92                     58.60
         64         12.28                52.34               58.24                     57.91
        128         10.29                54.82               58.75                     58.34
        256          8.96                56.30               58.46                     58.37


Badanie funkcji aktywacji 

In [26]:

# --- DEFINICJE NOWYCH FUNKCJI AKTYWACJI ---
class Activation_Sigmoid:
    def forward(self, inputs):
        self.inputs = inputs
        # Clipowanie zabezpiecza przed przepełnieniem matematycznym (np. funkcja exp)
        self.output = 1 / (1 + np.exp(-np.clip(inputs, -250, 250)))
    def backward(self, dvalues):
        self.dinputs = dvalues * (self.output * (1 - self.output))

class Activation_Tanh:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.tanh(inputs)
    def backward(self, dvalues):
        self.dinputs = dvalues * (1 - self.output ** 2)

class Activation_LeakyReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0.01 * inputs, inputs) # Puszcza 1% na ujemnych
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] *= 0.01

# --- USTAWIENIA BADANIA ---
# Zamrażamy najlepsze wartości z poprzednich kroków:
n1, n2 = 256, 128
najlepszy_lr = 0.01
najlepszy_batch = 64

aktywacje_do_testu = [
    ("ReLU", Activation_ReLU, Activation_ReLU),
    ("Leaky ReLU", Activation_LeakyReLU, Activation_LeakyReLU),
    ("Tanh", Activation_Tanh, Activation_Tanh),
    ("Sigmoid", Activation_Sigmoid, Activation_Sigmoid)
]

powtorzenia = 3       
epoki = 100           

wyniki_raport_act = []
liczba_cech = X_train.shape[1]

print("="*75)
print(f"ROZPOCZYNAMY BADANIE: Funkcja Aktywacji (Sieć: [{n1}, {n2}], LR: {najlepszy_lr}, Metryka: sMAPE)")
print("="*75)

for nazwa, AktKlasa1, AktKlasa2 in aktywacje_do_testu:
    print(f"\n---> Testuję: {nazwa:10} Pętla powtórzeń: ", end="")
    
    smape_train_historia = []
    smape_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        # 1. INICJUJEMY SIEĆ Z ZADANĄ FUNKCJĄ
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = AktKlasa1()  
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = AktKlasa2()  
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear() 
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 
        
        # 2. TRENING
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), najlepszy_batch):
                end_idx = start_idx + najlepszy_batch
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                # Forward
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                # Backward
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                # Aktualizacja wag
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # 3. EWALUACJA (Train)
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        
        licznik_tr = np.abs(prawdziwe_train_dolary - wymyslone_train_dolary)
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + np.abs(wymyslone_train_dolary)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # 4. EWALUACJA (Test)
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        
        licznik_te = np.abs(prawdziwe_test_dolary - wymyslone_test_dolary)
        mianownik_te = (np.abs(prawdziwe_test_dolary) + np.abs(wymyslone_test_dolary)) / 2.0
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100
        smape_test_historia.append(smape_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    sredni_czas_proby = czas_trwania / powtorzenia

    # 5. AGREGACJA WYNIKÓW
    wyniki_raport_act.append({
        "Funkcja Aktywacji": nazwa,
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

# --- WYSWIETLANIE TABELKI ---
print("\n" + "="*75)
print("PODSUMOWANIE WYNIKÓW: WPŁYW FUNKCJI AKTYWACJI")
print("="*75)
df_raport_act = pd.DataFrame(wyniki_raport_act)
print(df_raport_act.to_string(index=False))

ROZPOCZYNAMY BADANIE: Funkcja Aktywacji (Sieć: [256, 128], LR: 0.01, Metryka: sMAPE)

---> Testuję: ReLU       Pętla powtórzeń: 1... 2... 3... Zakończono w 64.1 s.

---> Testuję: Leaky ReLU Pętla powtórzeń: 1... 2... 3... Zakończono w 88.6 s.

---> Testuję: Tanh       Pętla powtórzeń: 1... 2... 3... Zakończono w 58.8 s.

---> Testuję: Sigmoid    Pętla powtórzeń: 1... 2... 3... Zakończono w 64.2 s.

PODSUMOWANIE WYNIKÓW: WPŁYW FUNKCJI AKTYWACJI
Funkcja Aktywacji  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
             ReLU         21.35                52.43               58.30                     58.03
       Leaky ReLU         29.53                52.44               58.26                     58.20
             Tanh         19.58                62.03               62.76                     62.56
          Sigmoid         21.40                68.19               68.03                     68.00


Badanie Epok

In [23]:

n1, n2 = 256, 128
najlepszy_lr = 0.01
najlepszy_batch = 64 

epoki_do_testu = [50, 100, 150, 200, 300]
powtorzenia = 3       

wyniki_raport_epoki = []
liczba_cech = X_train.shape[1]

print("="*75)
print(f"ROZPOCZYNAMY BADANIE: Liczba Epok (Sieć: [{n1}, {n2}], LR: {najlepszy_lr}, Batch: {najlepszy_batch})")
print("="*75)

for e in epoki_do_testu:
    print(f"\n---> Testuję: {e:3} epok Pętla powtórzeń: ", end="")
    
    smape_train_historia = []
    smape_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        # 1. INICJUJEMY SIEĆ
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU() 
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear() 
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 
        
        # 2. TRENING Z ZADANĄ LICZBĄ EPOK
        for epoch in range(e): 
            for start_idx in range(0, len(X_train), najlepszy_batch):
                end_idx = start_idx + najlepszy_batch
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # 3. EWALUACJA (Train)
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        
        licznik_tr = np.abs(prawdziwe_train_dolary - wymyslone_train_dolary)
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + np.abs(wymyslone_train_dolary)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # 4. EWALUACJA (Test)
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        
        licznik_te = np.abs(prawdziwe_test_dolary - wymyslone_test_dolary)
        mianownik_te = (np.abs(prawdziwe_test_dolary) + np.abs(wymyslone_test_dolary)) / 2.0
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100
        smape_test_historia.append(smape_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    sredni_czas_proby = czas_trwania / powtorzenia

    # 5. AGREGACJA WYNIKÓW
    wyniki_raport_epoki.append({
        "Liczba Epok": e,
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

# --- WYSWIETLANIE TABELKI ---
print("\n" + "="*75)
print("PODSUMOWANIE WYNIKÓW: WPŁYW LICZBY EPOK")
print("="*75)
df_raport_epoki = pd.DataFrame(wyniki_raport_epoki)
print(df_raport_epoki.to_string(index=False))

ROZPOCZYNAMY BADANIE: Liczba Epok (Sieć: [256, 128], LR: 0.01, Batch: 64)

---> Testuję:  50 epok Pętla powtórzeń: 1... 2... 3... Zakończono w 18.5 s.

---> Testuję: 100 epok Pętla powtórzeń: 1... 2... 3... Zakończono w 36.9 s.

---> Testuję: 150 epok Pętla powtórzeń: 1... 2... 3... Zakończono w 56.8 s.

---> Testuję: 200 epok Pętla powtórzeń: 1... 2... 3... Zakończono w 74.2 s.

---> Testuję: 300 epok Pętla powtórzeń: 1... 2... 3... Zakończono w 112.3 s.

PODSUMOWANIE WYNIKÓW: WPŁYW LICZBY EPOK
 Liczba Epok  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
          50          6.18                55.47               58.78                     58.49
         100         12.30                52.72               58.58                     58.17
         150         18.92                50.36               58.12                     57.99
         200         24.75                49.13               58.50                     58.19
         300         37.44   

Badanie Wielkości Dropout

In [ ]:

# --- USTAWIENIA BADANIA ---
# Zamrażamy najlepsze parametry z dotychczasowych badań:
n1, n2 = 256, 128
najlepszy_lr = 0.01
najlepszy_batch = 64
epoki = 150
powtorzenia = 3       

dropouty_do_testu = [0.0, 0.2, 0.3, 0.4, 0.5, 0.8]
wyniki_raport_drop = []
liczba_cech = X_train.shape[1]

print("="*75)
print(f"ROZPOCZYNAMY BADANIE: Współczynnik Dropout (Sieć: [{n1}, {n2}], LR: {najlepszy_lr})")
print("="*75)

for drop_rate in dropouty_do_testu:
    print(f"\n---> Testuję: Dropout {drop_rate} (Pętla powtórzeń: ", end="")
    
    smape_train_historia = []
    smape_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        # 1. INICJUJEMY SIEĆ Z ZADANYM DROPOUTEM
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        dropout1 = Layer_Dropout(drop_rate) 
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        dropout2 = Layer_Dropout(drop_rate)
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear() 
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 
        
        # 2. TRENING
        for epoch in range(epoki): 
            for start_idx in range(0, len(X_train), najlepszy_batch):
                end_idx = start_idx + najlepszy_batch
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                # Forward (Pamiętamy o flagach training=True!)
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dropout1.forward(activation1.output, training=True)
                
                dense2.forward(dropout1.output)
                activation2.forward(dense2.output)
                dropout2.forward(activation2.output, training=True)
                
                dense3.forward(dropout2.output)
                activation3.forward(dense3.output)
                
                # Backward
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                dropout2.backward(dense3.dinputs)
                activation2.backward(dropout2.dinputs)
                dense2.backward(activation2.dinputs)
                dropout1.backward(dense2.dinputs)
                activation1.backward(dropout1.dinputs)
                dense1.backward(activation1.dinputs)
                
                # Update
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # 3. EWALUACJA (Train) - Ważne: training=False, wyłączamy Dropout do testów!
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dropout1.forward(activation1.output, training=False)
        dense2.forward(dropout1.output)
        activation2.forward(dense2.output)
        dropout2.forward(activation2.output, training=False)
        dense3.forward(dropout2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        
        licznik_tr = np.abs(prawdziwe_train_dolary - wymyslone_train_dolary)
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + np.abs(wymyslone_train_dolary)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # 4. EWALUACJA (Test) - Ważne: training=False
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dropout1.forward(activation1.output, training=False)
        dense2.forward(dropout1.output)
        activation2.forward(dense2.output)
        dropout2.forward(activation2.output, training=False)
        dense3.forward(dropout2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        
        licznik_te = np.abs(prawdziwe_test_dolary - wymyslone_test_dolary)
        mianownik_te = (np.abs(prawdziwe_test_dolary) + np.abs(wymyslone_test_dolary)) / 2.0
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100
        smape_test_historia.append(smape_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    sredni_czas_proby = czas_trwania / powtorzenia

    # 5. AGREGACJA WYNIKÓW
    wyniki_raport_drop.append({
        "Współczynnik Dropout": drop_rate,
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

# --- WYSWIETLANIE TABELKI ---
print("\n" + "="*75)
print("PODSUMOWANIE WYNIKÓW: WPŁYW REGULARYZACJI DROPOUT")
print("="*75)
df_raport_drop = pd.DataFrame(wyniki_raport_drop)
print(df_raport_drop.to_string(index=False))

ROZPOCZYNAMY BADANIE: Współczynnik Dropout (Sieć: [256, 128], LR: 0.01)

---> Testuję: Dropout 0.0 (Pętla powtórzeń: 1... 2... 3... Zakończono w 120.1 s.

---> Testuję: Dropout 0.3 (Pętla powtórzeń: 1... 2... 3... Zakończono w 152.9 s.

---> Testuję: Dropout 0.4 (Pętla powtórzeń: 1... 2... 3... Zakończono w 164.7 s.

---> Testuję: Dropout 0.45 (Pętla powtórzeń: 1... 2... 3... Zakończono w 173.4 s.

PODSUMOWANIE WYNIKÓW: WPŁYW REGULARYZACJI DROPOUT
 Współczynnik Dropout  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
                 0.00         40.04                50.72               58.43                     58.33
                 0.30         50.97                52.34               57.05                     56.62
                 0.40         54.91                53.32               57.53                     56.67
                 0.45         57.80                53.07               57.01                     56.70


Badanie wielkości próby 

In [27]:
print("="*75)
print("EKSPERYMENT: WPŁYW WIELKOŚCI PRÓBY TRENINGOWEJ (LEARNING CURVE)")
print("="*75)

# Testujemy podziały: od 50% danych treningowych do 90%
podzialy = [0.5, 0.6, 0.7, 0.8, 0.9]
powtorzenia = 3
wyniki_krzywej = []

# Używamy zmiennych bazowych
zmienne_liczbowe_eksperyment = ["TOTAL DIMENSIONS", "YEAR", "Colorfulness Score", "SVD Entropy"]

for frac in podzialy:
    print(f"\n---> Testuję podział treningowy: {frac*100:.0f}% / testowy: {(1-frac)*100:.0f}% Pętla powtórzeń: ", end="")
    
    start_time = time.time()
    
    smape_train_historia = []
    smape_test_historia = []
    
    # -----------------------------------------------------
    # ETAP 1: BEZPIECZNE PRZYGOTOWANIE DANYCH 
    # -----------------------------------------------------
    # Losujemy zbiór (dodajemy random_state, by wyniki były powtarzalne)
    df_train_exp = df.sample(frac=frac, random_state=42)
    df_test_exp = df.drop(df_train_exp.index)
    
    # Wyliczamy statystyki TYLKO na zbiorze treningowym (Brak wycieku danych!)
    srednia_exp = df_train_exp[zmienne_liczbowe_eksperyment].mean()
    odchylenie_exp = df_train_exp[zmienne_liczbowe_eksperyment].std()
    
    X_num_train_exp = ((df_train_exp[zmienne_liczbowe_eksperyment] - srednia_exp) / odchylenie_exp).to_numpy(dtype=np.float32)
    X_num_test_exp = ((df_test_exp[zmienne_liczbowe_eksperyment] - srednia_exp) / odchylenie_exp).to_numpy(dtype=np.float32)
    
    indeksy_train_exp = df_train_exp.index
    indeksy_test_exp = df_test_exp.index
    
    X_kat_train_exp = X_kat_cale[indeksy_train_exp]
    X_kat_test_exp = X_kat_cale[indeksy_test_exp]
    
    X_train_gotowe = np.hstack([X_kat_train_exp, X_num_train_exp], dtype=np.float32)
    X_test_gotowe = np.hstack([X_kat_test_exp, X_num_test_exp], dtype=np.float32)
    
    y_train_gotowe = np.log(df_train_exp['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)
    y_test_gotowe = np.log(df_test_exp['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)
    
    liczba_cech = X_train_gotowe.shape[1]
    
    # -----------------------------------------------------
    # ETAP 2: TRENING SIECI
    # -----------------------------------------------------
    for p in range(powtorzenia):
        print(f"{p+1}...", end=" ")
        
        # OSTATECZNE, NAJLEPSZE PARAMETRY SIECI
        n1, n2 = 256, 128
        najlepszy_lr = 0.01
        najlepszy_batch = 64
        epoki = 150
        poziom_dropoutu = 0.3  
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        dropout1 = Layer_Dropout(poziom_dropoutu) 

        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        dropout2 = Layer_Dropout(poziom_dropoutu)

        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()

        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr)

        for epoch in range(epoki):
            for start_idx in range(0, len(X_train_gotowe), najlepszy_batch):
                end_idx = start_idx + najlepszy_batch
                X_batch = X_train_gotowe[start_idx:end_idx]
                y_batch = y_train_gotowe[start_idx:end_idx]
                
                # Przód sieci z włączonym Dropoutem
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dropout1.forward(activation1.output, training=True)
                
                dense2.forward(dropout1.output)
                activation2.forward(dense2.output)
                dropout2.forward(activation2.output, training=True)
                
                dense3.forward(dropout2.output)
                activation3.forward(dense3.output)
                
                # Tył sieci
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                dropout2.backward(dense3.dinputs)
                activation2.backward(dropout2.dinputs)
                dense2.backward(activation2.dinputs)
                dropout1.backward(dense2.dinputs)
                activation1.backward(dropout1.dinputs)
                dense1.backward(activation1.dinputs)
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # Ewaluacja (Train) - WYŁĄCZONY Dropout
        dense1.forward(X_train_gotowe)
        activation1.forward(dense1.output)
        dropout1.forward(activation1.output, training=False) 
        dense2.forward(dropout1.output)
        activation2.forward(dense2.output)
        dropout2.forward(activation2.output, training=False) 
        dense3.forward(dropout2.output)
        activation3.forward(dense3.output)

        wymyslone_ceny_train = prawdziwa_cena(activation3.output)
        prawdziwe_ceny_train = prawdziwa_cena(y_train_gotowe)

        licznik_train = np.abs(prawdziwe_ceny_train - wymyslone_ceny_train)
        mianownik_train = (np.abs(prawdziwe_ceny_train) + np.abs(wymyslone_ceny_train)) / 2.0
        smape_train = np.mean(licznik_train / (mianownik_train + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # Ewaluacja (Test) - WYŁĄCZONY Dropout
        dense1.forward(X_test_gotowe)
        activation1.forward(dense1.output)
        dropout1.forward(activation1.output, training=False) 
        dense2.forward(dropout1.output)
        activation2.forward(dense2.output)
        dropout2.forward(activation2.output, training=False) 
        dense3.forward(dropout2.output)
        activation3.forward(dense3.output)

        wymyslone_ceny_test = prawdziwa_cena(activation3.output)
        prawdziwe_ceny_test = prawdziwa_cena(y_test_gotowe)

        licznik_test = np.abs(prawdziwe_ceny_test - wymyslone_ceny_test)
        mianownik_test = (np.abs(prawdziwe_ceny_test) + np.abs(wymyslone_ceny_test)) / 2.0
        smape_test = np.mean(licznik_test / (mianownik_test + 1e-8)) * 100
        smape_test_historia.append(smape_test)

    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")
    
    sredni_czas_proby = czas_trwania / powtorzenia

    wyniki_krzywej.append({
        "Podział (Train/Test)": f"{frac*100:.0f}% / {(1-frac)*100:.0f}%",
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

print("\n" + "="*75)
print("PODSUMOWANIE WYNIKÓW: KRZYWA UCZENIA (LEARNING CURVE)")
print("="*75)
df_raport_krzywa = pd.DataFrame(wyniki_krzywej)
print(df_raport_krzywa.to_string(index=False))

EKSPERYMENT: WPŁYW WIELKOŚCI PRÓBY TRENINGOWEJ (LEARNING CURVE)

---> Testuję podział treningowy: 50% / testowy: 50% Pętla powtórzeń: 1... 2... 3... Zakończono w 101.5 s.

---> Testuję podział treningowy: 60% / testowy: 40% Pętla powtórzeń: 1... 2... 3... Zakończono w 118.8 s.

---> Testuję podział treningowy: 70% / testowy: 30% Pętla powtórzeń: 1... 2... 3... Zakończono w 147.0 s.

---> Testuję podział treningowy: 80% / testowy: 20% Pętla powtórzeń: 1... 2... 3... Zakończono w 157.1 s.

---> Testuję podział treningowy: 90% / testowy: 10% Pętla powtórzeń: 1... 2... 3... Zakończono w 172.9 s.

PODSUMOWANIE WYNIKÓW: KRZYWA UCZENIA (LEARNING CURVE)
Podział (Train/Test)  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
           50% / 50%         33.84                50.96               58.53                     58.19
           60% / 40%         39.59                51.38               58.14                     57.47
           70% / 30%         49.00      

Badanie wpływu SVD 

In [28]:

n1, n2 = 256, 128
najlepszy_lr = 0.01
najlepszy_batch = 64
epoki = 150        
powtorzenia = 3    

wyniki_raport_svd = []

print("="*75)
print(f"ROZPOCZYNAMY BADANIE: Wpływ Entropii SVD (Sieć: [{n1}, {n2}], LR: {najlepszy_lr}, Epoki: {epoki})")
print("="*75)

# Definiujemy warianty danych: A (Pełne z SVD), B (Bez ostatniej kolumny, czyli bez SVD)
warianty_do_testu = [
    ("Z entropią SVD (Pełne dane)", X_train, X_test),
    ("Bez entropii SVD", X_train[:, :-1], X_test[:, :-1])
]

for nazwa_wariantu, X_tr, X_te in warianty_do_testu:
    print(f"\n---> Testuję: {nazwa_wariantu:25} Pętla powtórzeń: ", end="")
    
    smape_train_historia = []
    smape_test_historia = []
    
    start_time = time.time()
    liczba_cech = X_tr.shape[1]
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        # 1. INICJUJEMY SIEĆ Z WARSTWAMI DROPOUT
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        dropout1 = Layer_Dropout(0.3) 

        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        dropout2 = Layer_Dropout(0.3)

        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()

        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr)
        
        # 2. TRENING
        for epoch in range(epoki):
            for start_idx in range(0, len(X_tr), najlepszy_batch):
                end_idx = start_idx + najlepszy_batch
                X_batch = X_tr[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                # Forward (z training=True dla Dropoutu)
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dropout1.forward(activation1.output, training=True)
                
                dense2.forward(dropout1.output)
                activation2.forward(dense2.output)
                dropout2.forward(activation2.output, training=True)
                
                dense3.forward(dropout2.output)
                activation3.forward(dense3.output)
                
                # Backward
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                dropout2.backward(dense3.dinputs)
                activation2.backward(dropout2.dinputs)
                dense2.backward(activation2.dinputs)
                dropout1.backward(dense2.dinputs)
                activation1.backward(dropout1.dinputs)
                dense1.backward(activation1.dinputs)
                
                # Update
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # 3. EWALUACJA (Train) - Dropout wyłączony (training=False)
        dense1.forward(X_tr)
        activation1.forward(dense1.output)
        dropout1.forward(activation1.output, training=False)
        dense2.forward(dropout1.output)
        activation2.forward(dense2.output)
        dropout2.forward(activation2.output, training=False)
        dense3.forward(dropout2.output)
        activation3.forward(dense3.output)

        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        
        licznik_tr = np.abs(prawdziwe_train_dolary - wymyslone_train_dolary)
        mianownik_tr = (np.abs(prawdziwe_train_dolary) + np.abs(wymyslone_train_dolary)) / 2.0
        smape_train = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100
        smape_train_historia.append(smape_train)

        # 4. EWALUACJA (Test) - Dropout wyłączony (training=False)
        dense1.forward(X_te)
        activation1.forward(dense1.output)
        dropout1.forward(activation1.output, training=False)
        dense2.forward(dropout1.output)
        activation2.forward(dense2.output)
        dropout2.forward(activation2.output, training=False)
        dense3.forward(dropout2.output)
        activation3.forward(dense3.output)

        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        
        licznik_te = np.abs(prawdziwe_test_dolary - wymyslone_test_dolary)
        mianownik_te = (np.abs(prawdziwe_test_dolary) + np.abs(wymyslone_test_dolary)) / 2.0
        smape_test = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100
        smape_test_historia.append(smape_test)

    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    sredni_czas_proby = czas_trwania / powtorzenia

    # 5. AGREGACJA
    wyniki_raport_svd.append({
        "Wariant Danych": nazwa_wariantu,
        "Śr. Czas [s]": round(sredni_czas_proby, 2),
        "Śr. sMAPE Train [%]": round(np.mean(smape_train_historia), 2),
        "Śr. sMAPE Test [%]": round(np.mean(smape_test_historia), 2),
        "Najlepsze sMAPE Test [%]": round(np.min(smape_test_historia), 2)
    })

# --- WYSWIETLANIE TABELKI ---
print("\n" + "="*75)
print("PODSUMOWANIE WYNIKÓW: WPŁYW ENTROPII SVD NA JAKOŚĆ MODELU")
print("="*75)
df_raport_svd = pd.DataFrame(wyniki_raport_svd)
print(df_raport_svd.to_string(index=False))

ROZPOCZYNAMY BADANIE: Wpływ Entropii SVD (Sieć: [256, 128], LR: 0.01, Epoki: 150)

---> Testuję: Z entropią SVD (Pełne dane) Pętla powtórzeń: 1... 2... 3... Zakończono w 157.0 s.

---> Testuję: Bez entropii SVD          Pętla powtórzeń: 1... 2... 3... Zakończono w 156.4 s.

PODSUMOWANIE WYNIKÓW: WPŁYW ENTROPII SVD NA JAKOŚĆ MODELU
             Wariant Danych  Śr. Czas [s]  Śr. sMAPE Train [%]  Śr. sMAPE Test [%]  Najlepsze sMAPE Test [%]
Z entropią SVD (Pełne dane)         52.34                53.04               57.81                     56.64
           Bez entropii SVD         52.12                52.93               57.25                     57.15


# Model Końcowy

In [33]:
print("="*75)
print("TRENOWANIE OSTATECZNEGO MODELU (ŚREDNIA Z 3 PRÓB, BEZ SVD ENTROPY)")
print("="*75)

# 1. Usuwamy SVD Entropy (odcinamy ostatnią kolumnę)
X_train_final = X_train[:, :-1]
X_test_final = X_test[:, :-1]
liczba_cech = X_train_final.shape[1]

# 2. Najlepsze wyznaczone hiperparametry z całego badania
n1, n2 = 256, 128
najlepszy_lr = 0.01
najlepszy_batch = 64
epoki = 150 
poziom_dropoutu = 0.3
powtorzenia = 3

# Listy do przechowywania wyników z 3 prób
train_mae_list, train_rmse_list, train_r2_list, train_mape_list, train_smape_list = [], [], [], [], []
test_mae_list, test_rmse_list, test_r2_list, test_mape_list, test_smape_list = [], [], [], [], []

start_time = time.time()

# 3. Pętla uśredniająca eksperyment
for p in range(powtorzenia):
    print(f"---> Uruchamianie próby {p+1}/{powtorzenia}...")
    
    # Inicjalizacja optymalnej sieci (włączony Dropout)
    dense1 = Layer_Dense(liczba_cech, n1)  
    activation1 = Activation_ReLU()
    dropout1 = Layer_Dropout(poziom_dropoutu)  

    dense2 = Layer_Dense(n1, n2)  
    activation2 = Activation_ReLU()
    dropout2 = Layer_Dropout(poziom_dropoutu)  

    dense3 = Layer_Dense(n2, 1)  
    activation3 = Activation_Linear()

    loss_function = Loss_MSE()
    optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 

    # Klasyczna Pętla Treningowa
    for epoch in range(epoki):
        for start_idx in range(0, len(X_train_final), najlepszy_batch):
            end_idx = start_idx + najlepszy_batch
            
            X_batch = X_train_final[start_idx:end_idx]
            y_batch = y_train[start_idx:end_idx]
            
            # Przepływ w przód
            dense1.forward(X_batch)
            activation1.forward(dense1.output)
            dropout1.forward(activation1.output, training=True)  
            
            dense2.forward(dropout1.output)
            activation2.forward(dense2.output)
            dropout2.forward(activation2.output, training=True)  
            
            dense3.forward(dropout2.output)
            activation3.forward(dense3.output)
            
            # Obliczanie straty i przepływ wsteczny
            loss_function.backward(activation3.output, y_batch)
            
            activation3.backward(loss_function.dinputs)
            dense3.backward(activation3.dinputs)
            
            dropout2.backward(dense3.dinputs)
            activation2.backward(dropout2.dinputs)
            dense2.backward(activation2.dinputs)
            
            dropout1.backward(dense2.dinputs)
            activation1.backward(dropout1.dinputs)
            dense1.backward(activation1.dinputs)
            
            # Aktualizacja wag
            optimizer.update_params(dense1)
            optimizer.update_params(dense2)
            optimizer.update_params(dense3)

    # ==========================================
    # EWALUACJA NA ZBIORZE TRENINGOWYM
    # ==========================================
    dense1.forward(X_train_final)
    activation1.forward(dense1.output)
    dropout1.forward(activation1.output, training=False) 
    
    dense2.forward(dropout1.output)
    activation2.forward(dense2.output)
    dropout2.forward(activation2.output, training=False) 
        
    dense3.forward(dropout2.output)
    activation3.forward(dense3.output)

    wymyslone_ceny_train = prawdziwa_cena_log(activation3.output)
    prawdziwe_ceny_train = prawdziwa_cena_log(y_train)

    # Obliczenia metryk dla Treningu
    mae_tr = np.mean(np.abs(prawdziwe_ceny_train - wymyslone_ceny_train))
    rmse_tr = np.sqrt(np.mean((prawdziwe_ceny_train - wymyslone_ceny_train)**2))
    ss_res_tr = np.sum((prawdziwe_ceny_train - wymyslone_ceny_train)**2)
    ss_tot_tr = np.sum((prawdziwe_ceny_train - np.mean(prawdziwe_ceny_train))**2)
    r2_tr = 1 - (ss_res_tr / ss_tot_tr)

    non_zero_tr = prawdziwe_ceny_train != 0
    mape_tr = np.mean(np.abs((prawdziwe_ceny_train[non_zero_tr] - wymyslone_ceny_train[non_zero_tr]) / prawdziwe_ceny_train[non_zero_tr])) * 100

    licznik_tr = np.abs(prawdziwe_ceny_train - wymyslone_ceny_train)
    mianownik_tr = (np.abs(prawdziwe_ceny_train) + np.abs(wymyslone_ceny_train)) / 2.0
    smape_tr = np.mean(licznik_tr / (mianownik_tr + 1e-8)) * 100

    train_mae_list.append(mae_tr)
    train_rmse_list.append(rmse_tr)
    train_r2_list.append(r2_tr)
    train_mape_list.append(mape_tr)
    train_smape_list.append(smape_tr)

    # ==========================================
    # EWALUACJA NA ZBIORZE TESTOWYM
    # ==========================================
    dense1.forward(X_test_final)
    activation1.forward(dense1.output)
    dropout1.forward(activation1.output, training=False) 
    
    dense2.forward(dropout1.output)
    activation2.forward(dense2.output)
    dropout2.forward(activation2.output, training=False) 
    
    dense3.forward(dropout2.output)
    activation3.forward(dense3.output)

    wymyslone_ceny_test = prawdziwa_cena_log(activation3.output)
    prawdziwe_ceny_test = prawdziwa_cena_log(y_test)

    # Obliczenia metryk dla Testu
    mae_te = np.mean(np.abs(prawdziwe_ceny_test - wymyslone_ceny_test))
    rmse_te = np.sqrt(np.mean((prawdziwe_ceny_test - wymyslone_ceny_test)**2))
    ss_res_te = np.sum((prawdziwe_ceny_test - wymyslone_ceny_test)**2)
    ss_tot_te = np.sum((prawdziwe_ceny_test - np.mean(prawdziwe_ceny_test))**2)
    r2_te = 1 - (ss_res_te / ss_tot_te)

    non_zero_te = prawdziwe_ceny_test != 0
    mape_te = np.mean(np.abs((prawdziwe_ceny_test[non_zero_te] - wymyslone_ceny_test[non_zero_te]) / prawdziwe_ceny_test[non_zero_te])) * 100

    licznik_te = np.abs(prawdziwe_ceny_test - wymyslone_ceny_test)
    mianownik_te = (np.abs(prawdziwe_ceny_test) + np.abs(wymyslone_ceny_test)) / 2.0
    smape_te = np.mean(licznik_te / (mianownik_te + 1e-8)) * 100

    test_mae_list.append(mae_te)
    test_rmse_list.append(rmse_te)
    test_r2_list.append(r2_te)
    test_mape_list.append(mape_te)
    test_smape_list.append(smape_te)

czas_calkowity = time.time() - start_time

# ==========================================
# WYŚWIETLANIE RAPORTÓW SZCZEGÓŁOWYCH (DO PORÓWNANIA Z BAZOWYM)
# ==========================================
print(f"\nUkończono w {czas_calkowity:.1f} sekund.")

print("\n" + "-"*60)
print(f"SZCZEGÓŁOWY RAPORT: DANE TRENINGOWE (ŚREDNIA Z {powtorzenia} PRÓB)")
print("-"  * 60)
print(f"1. MAE:       {np.mean(train_mae_list):10.2f} $")
print(f"2. RMSE:      {np.mean(train_rmse_list):10.2f} $")
print(f"3. R^2:       {np.mean(train_r2_list):10.4f}")
print(f"4. MAPE:      {np.mean(train_mape_list):10.2f} %")
print(f"5. sMAPE:     {np.mean(train_smape_list):10.2f} %")

print("\n" + "-"*60)
print("SZCZEGÓŁOWY RAPORT: DANE TESTOWE (WYNIK OFICJALNY)")
print("-" * 60)
print(f"Dane:         Logarytmiczne, BEZ SVD")
print(f"Architektura: Wejście ({liczba_cech}) -> {n1} -> Drop({poziom_dropoutu}) -> {n2} -> Drop({poziom_dropoutu}) -> 1")
print(f"Trening:      LR = {najlepszy_lr}, Batch = {najlepszy_batch}, Epoki = {epoki}")
print("-" * 60)
print(f"1. MAE:       {np.mean(test_mae_list):10.2f} $")
print(f"2. RMSE:      {np.mean(test_rmse_list):10.2f} $")
print(f"3. R^2:       {np.mean(test_r2_list):10.4f}")
print(f"4. MAPE:      {np.mean(test_mape_list):10.2f} %")
print(f"5. sMAPE:     {np.mean(test_smape_list):10.2f} %")

# ==========================================
# WYŚWIETLANIE TABELI PODSUMOWUJĄCEJ (DO PORÓWNANIA Z PARAMETRAMI)
# ==========================================
df_raport_finalny = pd.DataFrame([{
    "Model": "Ostateczny (Bez SVD, Dropout 0.3)",
    "Śr. Czas [s]": round(czas_calkowity / powtorzenia, 2),
    "Śr. sMAPE Train [%]": round(np.mean(train_smape_list), 2),
    "Śr. sMAPE Test [%]": round(np.mean(test_smape_list), 2),
    "Najlepsze sMAPE Test [%]": round(np.min(test_smape_list), 2)
}])

print("\n" + "="*75)
print("TABELA PODSUMOWUJĄCA WYNIK FINAŁOWY")
print("="*75)
print(df_raport_finalny.to_string(index=False))

TRENOWANIE OSTATECZNEGO MODELU (ŚREDNIA Z 3 PRÓB, BEZ SVD ENTROPY)
---> Uruchamianie próby 1/3...
---> Uruchamianie próby 2/3...
---> Uruchamianie próby 3/3...

Ukończono w 189.6 sekund.

------------------------------------------------------------
SZCZEGÓŁOWY RAPORT: DANE TRENINGOWE (ŚREDNIA Z 3 PRÓB)
------------------------------------------------------------
1. MAE:            99.09 $
2. RMSE:          327.87 $
3. R^2:           0.5355
4. MAPE:          122.63 %
5. sMAPE:          52.82 %

------------------------------------------------------------
SZCZEGÓŁOWY RAPORT: DANE TESTOWE (WYNIK OFICJALNY)
------------------------------------------------------------
Dane:         Logarytmiczne, BEZ SVD
Architektura: Wejście (414) -> 256 -> Drop(0.3) -> 128 -> Drop(0.3) -> 1
Trening:      LR = 0.01, Batch = 64, Epoki = 150
------------------------------------------------------------
1. MAE:           107.13 $
2. RMSE:          338.20 $
3. R^2:           0.4830
4. MAPE:          175.79 %
5.